In [1]:
import os
os.chdir('/workspace/0ae9609e-50ff-44f0-85e8-67b2e92c28ae')
print(os.listdir('.'))


['-v8-validated-data.md', 'memory', '_zeros.py', '.prompts', '.config', 'weil_quadratic_form_general.py', 'weil_modular.py', '.kernel_llm_logs_1.txt']


In [2]:
import os
print(os.listdir('cache') if os.path.isdir('cache') else 'no cache dir')


no cache dir


In [3]:

# Let me think carefully about the archimedean term derivation.
# 
# For L(Δ,s), the completed L-function is:
# Λ(Δ,s) = (2π)^{-s} Γ(s + 11/2) L(Δ,s) [in arithmetic normalization s_arith]
# Functional equation: Λ(Δ,s) = Λ(Δ,12-s) (degree 2, weight 12, level 1)
#
# In analytic normalization, we shift s -> s + (k-1)/2 = s + 11/2 so critical line becomes Re(s)=1/2.
# Then the gamma factor becomes Γ_C(s + (k-1)/2) but it's really written as:
# γ_∞(s) = (2π)^{-s} Γ(s + (k-1)/2) in analytic norm with Λ(s) = Λ(1-s)
#
# Alternatively, recall Γ_C(s) := 2(2π)^{-s}Γ(s), so:
# γ_∞(s) = Γ_C(s + (k-1)/2) / 2 up to constants
#
# Or use: Γ_C(s) = Γ_R(s) Γ_R(s+1) where Γ_R(s) = π^{-s/2}Γ(s/2)
# So Γ_C(s + 11/2) = Γ_R(s + 11/2) Γ_R(s + 13/2)
# = π^{-(s+11/2)/2}Γ((s+11/2)/2) · π^{-(s+13/2)/2}Γ((s+13/2)/2)
#
# In Weil's explicit formula:
# Φ(r) = log(c_∞) + (γ_∞')(½+ir)/γ_∞ + (γ_∞')(½-ir)/γ_∞
# where γ_∞(s) is the archimedean factor.
#
# For γ_∞(s) = Γ_C(s + 11/2) = 2(2π)^{-(s+11/2)}Γ(s+11/2):
# log γ_∞(s) = log 2 - (s+11/2)log(2π) + log Γ(s+11/2)
# γ_∞'(s)/γ_∞(s) = -log(2π) + ψ(s+11/2)
#
# So Φ(r) for the archimedean part is:
# -2 log(2π) + ψ(1/2+ir+11/2) + ψ(1/2-ir+11/2)
# = -2 log(2π) + ψ(6+ir) + ψ(6-ir)
# = -2 log(2π) + 2 Re ψ(6+ir)
# = -2 log(2π) + 2 Re ψ(k/2 + ir) [k=12]
#
# So actually the current implementation 2*Re ψ(k/2+ir) IS correct for this gamma factor decomposition!
# The factor of 2 comes from the two terms ψ(s+11/2) + ψ(1-s+11/2), NOT from doubling.
#
# Now, hypothesis says "Γ(s+11/2) and Γ(s-11/2) after a shift, suggesting Re[ψ(s+11/2) + ψ(s-11/2)]"
# But this doesn't match the standard explicit formula derivation for L(Δ).
# 
# Let me think again. The functional equation in symmetric form might be written:
# ξ(s) := (some prefactor) · Γ(s + 11/2) Γ(s + 11/2 - 11) L(Δ, s + 1/2 + 11/2)?
# No - that's wrong. Let me look more carefully.
#
# Actually, the issue might be the factor of 1/2 normalization in the digamma argument.
# Compare to Dirichlet: arg = (1/2+a)/2 + i r/2 -- note division by 2 in BOTH places!
# This is because for Dirichlet L, γ_∞(s) = π^{-(s+a)/2} Γ((s+a)/2), so:
# γ_∞'(s)/γ_∞(s) = -log(π)/2 + (1/2) ψ((s+a)/2)
#
# The half comes from chain rule on (s+a)/2.
# Then Φ(r) = log q - log π + (1/2)[ψ((1/2+a+ir)/2) + ψ((1/2+a-ir)/2)]
# = log(q/π) + Re ψ((1/2+a)/2 + ir/2) [NO leading 2 because the 1/2 absorbs it]
#
# That matches the dirichlet code.
#
# For modular: γ_∞(s) = (2π)^{-(s+11/2)} Γ(s+11/2). No 1/2 in argument!
# γ_∞'(s)/γ_∞(s) = -log(2π) + ψ(s+11/2)
# Φ(r) = -2 log(2π) + ψ(1/2+ir+11/2) + ψ(1/2-ir+11/2) = -2 log(2π) + 2 Re ψ(6+ir)
#
# OK so 2 Re ψ(6+ir) seems correct.
#
# BUT WAIT - maybe the issue is more subtle. The current weil_modular.py also divides by (2π) at the end:
# M_arch = (HP @ H.T) * exp_neg_norm / (2*pi)
# whereas a more careful comparison with weil_quadratic_form_general would be needed.
# Both Dirichlet and modular versions divide by (2π) so probably consistent.
#
# Note: in Dirichlet version, the digamma factor is `np.real(digamma(arg))`. The "2" from the sum
# ψ(...+ir) + ψ(...-ir) = 2 Re ψ(...) is absorbed by the half-factor on integration when you set
# r' = r/2... no, that's not right.
#
# Hmm, let me carefully redo the Dirichlet case:
# Φ(r) = log(q/π) + (1/2)[ψ((1/2+a+ir)/2) + ψ((1/2+a-ir)/2)]
# = log(q/π) + Re ψ((1/2+a)/2 + ir/2)
# The code uses:
# psi_vals = np.real(digamma(arg)) with arg = (0.5+a)/2 + 0.5j * t
# So psi_vals = Re ψ((1/2+a)/2 + i r/2) — matches!
#
# For modular:
# Φ(r) = -2 log(2π) + 2 Re ψ(k/2 + ir)
# The code uses:
# psi_vals = 2.0 * np.real(digamma((k/2.0) + 1j * t_nodes))
# So psi_vals = 2 Re ψ(k/2 + ir) — matches!
#
# Hmm, but the polar term contributions need to add too:
# Dirichlet: M_polar = log(q/π)/(2π) — and that's the constant log(q/π) ABOVE in Φ ✓
# Modular: polar_const = log(N) - 2 log(2π). For N=1: = -2 log(2π). ✓ This matches my -2 log(2π).
#
# So everything seems right in principle. Why a 4% discrepancy?
# 
# Wait - maybe the issue is in the prime sum. Let me check the Dirichlet:
# Dirichlet: M_primes = Σ χ(p^k) (log p)/√p^k · g_ij(k log p), divided by π, taking Re
# Modular: weights = c_k(p) log p / √p^k, then M_primes = sum w_n · Re g_ij(k log p) / π
#
# Looking at the Weil explicit formula for L(Δ):
# (1/2π) ∫ Φ(r) h(r) dr = Σ_ρ h(γ) + 2 Σ_{n≥1} [a_Δ(n) Λ(n)/√n] g(log n)
# Or: Σ_ρ h(γ) = (1/2π) ∫ Φ(r) h(r) dr - 2 Σ_{p,k} a_Δ(p^k) log(p)/p^{k/2} g(k log p)
# where a_Δ(p^k) at prime powers comes from log L = Σ a(n)/n^s : for L=Π(1-α_p p^{-s})^{-1}(1-β_p p^{-s})^{-1}:
# log L(s) = Σ_p Σ_{k≥1} (α_p^k + β_p^k)/k · p^{-ks}
# Differentiating: -L'/L(s) = Σ_p Σ_{k≥1} (α_p^k + β_p^k) log(p) p^{-ks}
# So the von Mangoldt-type sum is Σ_{p,k} c_k(p) log(p) p^{-ks} where c_k(p) = α_p^k + β_p^k.
#
# Weil formula then says (with appropriate sign):
# Σ_ρ h(γ) = arch + polar - 2 Σ_{p,k} c_k(p) log p / p^{k/2} · g(k log p)
# where g is the Fourier transform of h.
#
# Compare to ζ:
# Σ_ρ h(γ) = arch + polar - 2 Σ_{p,k} log p / p^{k/2} · g(k log p)
# i.e. c_k(p)=1 always (since 1+1+...; wait ζ has one Euler factor, so c_k(p)=1).
# So the factor "2" out front is suspicious.
#
# Hmm. For ζ: α=1, β=0 (one Euler factor (1-p^{-s})^{-1}), so c_k(p) = α^k = 1.
# But the explicit formula for ζ involves: -2 Σ Λ(n)/√n g(log n), because the formula uses h(r) at r and -r symmetrically.
# Actually the "2" coefficient is from doubling of the integral - the standard Weil formula uses h symmetric (h(r)=h(-r)).
# 
# Looking at the ζ engine: weights = log(p)/√p^k (no factor 2), and uses Re g (which is real anyway), divided by π.
# The g here is g_ij(u) = ∫ φ_i(r) φ_j(r) e^{-iur} dr.
# Note that (1/2π) ∫ Φ_∞(r) h(r) dr where h(r)=|F(r)|^2 = (φ_i*φ_j integrated against...).
# 
# Hmm, I think the engine has a specific convention. The fact that ζ and L(χ4) are validated 
# at this precision means the conventions work for degree 1.
#
# Let me just RUN the existing code and quantify the residual carefully, then test variants.

print("Analysis: examining the implementation carefully")


Analysis: examining the implementation carefully


In [4]:

# Let me first run baseline: compute Q for L(Δ) using existing weil_modular.py at small parameters
# to establish the current ~4% residual, then test variants.

# Need zeros for L(Δ,s). Use _zeros.py but L_Delta_zeros is expensive.
# Try smaller N first, and see how many zeros are needed for a T0=85.7 sigma=2 window
# The window is roughly T0 ± 5σ = [75.7, 95.7] in absolute value, so we need zeros up to ~100+
# Looking up: L(Δ) zero density is roughly (log T)/(π) per unit T, but with 2 Γ-factors it's higher.
# Let's say we need ~50 zeros minimum for window coverage, but for accurate trace identity need more.

import sys
sys.path.insert(0, '.')
from _zeros import L_Delta_zeros
import time

# Try computing 20 zeros first to test speed
t0 = time.time()
zeros20 = L_Delta_zeros(20, dps=30, n_taus=100, y_max=10)
print(f"20 zeros in {time.time()-t0:.1f}s")
print("First 5:", [float(z) for z in zeros20[:5]])


20 zeros in 135.8s
First 5: [9.222379399921103, 13.907549861392134, 17.442776978234495, 19.65651314195495, 22.336103637209522]


In [5]:

# Good - first zero matches the expected 9.2223793999211025 ✓
# Need zeros up to T0+ several sigmas. With T0=85.7, sigma=2, we need zeros at least up to ~100-110
# Let me get more zeros - the cost ~T^4 means it'll get slow. Try 60 first.
t0 = time.time()
zeros60 = L_Delta_zeros(60, dps=30, n_taus=100, y_max=10)
print(f"60 zeros in {time.time()-t0:.1f}s")
print("Last few:", [float(z) for z in zeros60[-5:]])


60 zeros in 610.5s
Last few: [98.46428973852338, 99.826041663228, 101.18784753320837, 102.5497058899115, 103.91161531815692]


In [6]:

# Good. 60 zeros up to ~104. Need to extend further to fully include T0+5σ = 95.7 with margin.
# Actually 60 zeros covers up to 104. T0+5σ=95.7 so we're ok, but contributions from outside window decay 
# rapidly via Gaussian. Let's get a few more for safety - say up to gamma=120.
t0 = time.time()
zeros80 = L_Delta_zeros(80, dps=30, n_taus=100, y_max=10)
print(f"80 zeros in {time.time()-t0:.1f}s")
print("Last:", float(zeros80[-1]))


80 zeros in 880.8s
Last: 131.15868806597848


In [7]:

# 80 zeros up to ~131. T0=85.7 sigma=2 -> Gaussian decay e^(-((131-85.7)/2)^2/2) = e^(-256) tiny.
# 80 zeros is plenty.
# Now let's compute Q with the current weil_modular.py 
from weil_modular import compute_Q_modular
import numpy as np

zeros_float = np.array([float(z) for z in zeros80])

T0 = 85.7
sigma = 2.0
J = 10

t0 = time.time()
Q_orig, comps_orig = compute_Q_modular(
 zeros_float, T0, sigma, J,
 weight=12, conductor=1,
 primes_cutoff=100000, # X=10^5
 n_nodes_quad=200,
 include_negative_zeros=True,
)
print(f"Computed Q in {time.time()-t0:.1f}s")
print(f"tr(M_zeros) = {np.trace(comps_orig['M_zeros']):.10e}")
print(f"tr(M_arith) = {np.trace(comps_orig['M_arith']):.10e}")
print(f"tr(M_polar) = {np.trace(comps_orig['M_polar']):.10e}")
print(f"tr(M_arch) = {np.trace(comps_orig['M_arch']):.10e}")
print(f"tr(M_primes)= {np.trace(comps_orig['M_primes']):.10e}")
tr_diff = np.trace(comps_orig['M_zeros']) - np.trace(comps_orig['M_arith'])
print(f"\ntr(M_zeros) - tr(M_arith) = {tr_diff:.6e}")
print(f"relative: {tr_diff/np.trace(comps_orig['M_zeros']):.4%}")
eigs = np.linalg.eigvalsh(Q_orig)
print(f"\nλ_min(Q) = {eigs.min():.6e}")
print(f"|λ_min|/tr(M_zeros) = {abs(eigs.min())/np.trace(comps_orig['M_zeros']):.6e}")


TimeoutError: Code execution timed out after 900.0 seconds

In [8]:

# Timed out - probably because too many prime powers. Reduce primes_cutoff for fast iteration.
# Let me use X=10^4 first to debug, then push to 10^5 only for the final run.
import sys, importlib, time
sys.path.insert(0, '.')
import weil_modular
importlib.reload(weil_modular)
from weil_modular import compute_Q_modular
import numpy as np
import pickle, os

# Load cached zeros
with open('cache/LDelta_zeros_N80_dps30.pkl','rb') as f:
 zeros80 = pickle.load(f)
zeros_float = np.array([float(z) for z in zeros80])

T0, sigma, J = 85.7, 2.0, 10

t0 = time.time()
Q_orig, comps_orig = compute_Q_modular(
 zeros_float, T0, sigma, J,
 weight=12, conductor=1,
 primes_cutoff=10000,
 n_nodes_quad=100,
 include_negative_zeros=True,
)
print(f"X=1e4 nodes=100: {time.time()-t0:.1f}s")
trZ = np.trace(comps_orig['M_zeros'])
trA = np.trace(comps_orig['M_arith'])
print(f"tr(M_zeros) = {trZ:.10e}")
print(f"tr(M_arith) = {trA:.10e}")
print(f"diff = {trZ-trA:.6e} rel = {(trZ-trA)/trZ:.4%}")
print(f" M_polar tr={np.trace(comps_orig['M_polar']):.6e}")
print(f" M_arch tr={np.trace(comps_orig['M_arch']):.6e}")
print(f" M_primes tr={np.trace(comps_orig['M_primes']):.6e}")


X=1e4 nodes=100: 89.4s
tr(M_zeros) = 7.3900321742e+00
tr(M_arith) = 8.2839036150e+00
diff = -8.938714e-01 rel = -12.0956%
 M_polar tr=-5.850144e+00
 M_arch tr=1.416974e+01
 M_primes tr=3.568804e-02


In [9]:

# Big discrepancy ~12% — but with X=1e4. The original 4% was at X=1e5.
# Now test the hypothesis: archimedean uses 2*Re ψ(k/2 + ir) [current], 
# vs Re[ψ(s+11/2) + ψ(s-11/2)] meaning Re ψ(6+ir) + Re ψ(-5+ir)
# But ψ(-5+ir) has poles near integers... this doesn't make sense physically.
#
# Alternative interpretation: After the shift to analytic norm s -> s + (k-1)/2,
# functional eqn becomes Λ(s) = Λ(1-s) with Γ-factor Γ(s + (k-1)/2) · (...).
# The "two Γ factors" Γ(s+11/2) and Γ(s-11/2) come from rewriting the SYMMETRIC 
# completed L-function around s=1/2:
# Λ_sym(s) = γ_∞(s) γ_∞(1-s)^something... no this is double-counting.
# 
# Actually the cleanest derivation: γ_∞(s) for L(Δ) in analytic norm is (2π)^{-s} Γ(s + 11/2).
# The Weil explicit formula integrand on critical line s=1/2+ir is:
# d/ds log γ_∞(s) |_{s=1/2+ir} + d/ds log γ_∞(s) |_{s=1/2-ir} 
# = [-log(2π) + ψ(6+ir)] + [-log(2π) + ψ(6-ir)]
# = -2log(2π) + 2 Re ψ(6+ir)
# This is what's in the code. So mathematically the "factor of 2" is correct.
#
# Maybe the bug is elsewhere. Look at sign convention for prime sum.
# Spec note: "the sign of the prime sum in the explicit formula must be negative for the trace identity to hold"
# In compute_Q_modular: M_arith = M_polar + M_arch - M_primes
# Q = M_zeros - M_arith
# So tr(Q) ≈ 0 means tr(M_zeros) ≈ tr(M_polar + M_arch - M_primes)
# 
# Maybe the prime sum coefficient is wrong: standard EF has "−2 Σ c_k(p) log p / p^{k/2} g(k log p)"
# but the modular code accumulates weights = c_k(p) log p / sqrt(p^k) — NO factor of 2.
# For ζ engine: weights = log p / sqrt(p^k) — also no factor of 2.
# This works for ζ because... the integrand g_ij(u) already accounts for both r and -r symmetry?
# 
# Let me check: g_ij(u) = ∫ φ_i(r)φ_j(r) e^{-iur} dr
# Re g_ij(u) = ∫ φ_i(r) φ_j(r) cos(ur) dr — this is symmetric in u
# In EF: Σ c_k(p) log p/sqrt(p^k) [g(k log p) + g(-k log p)] = 2 Σ c_k(p) log p / sqrt(p^k) Re g(k log p)
# So the "2" is hidden in the use of Re g. OK so prime side is consistent with degree-2 EF.
#
# Let me reconsider: maybe M_polar / 2π double-counts something. The Dirichlet EF is:
# Σ_ρ h(γ) = (1/2π) ∫ h(r) [log(q/π) + Re ψ((1/2+a)/2 + ir/2)] dr - 2 Σ Re χ(n) Λ(n)/sqrt(n) g(log n)
# h(r) here, when h(r)=φ_i(r)φ_j(r), gives Σ_ρ φ_i(γ_ρ)φ_j(γ_ρ) on LHS (trace of M_zeros for diagonal).
# (1/2π) ∫ [log(q/π)] φ_i(r)^2 dr = log(q/π)/(2π) (since ∫φ_i^2=1) — agrees with M_polar.
# (1/2π) ∫ Re ψ(...+ir/2) φ_i^2 dr — this is the M_arch entry.
#
# For modular EF (degree 2 holomorphic cusp form, level N, weight k, no pole):
# Σ_ρ h(γ) = (1/2π) ∫ h(r) [log N - 2 log(2π) + 2 Re ψ(k/2+ir)] dr - 2 Σ a_f(n) Λ_f(n)/sqrt(n) g(log n)
# where Λ_f(n) = c_k(p) log p for n=p^k.
# (1/2π) ∫ h(r) [log N - 2 log(2π)] dr = (log N - 2 log 2π)/(2π) — matches M_polar.
# Prime sum: -2 Σ c_k(p) log p / sqrt(p^k) [Re g(k log p)] — matches code.
# Archimedean: (1/2π) ∫ h(r) · 2 Re ψ(k/2+ir) dr — matches code (factor 2.0 inside).
#
# So everything seems CORRECT. Why the residual?
# 
# Maybe the problem isn't the convention but rather:
# (a) Number of zeros (truncation error) — but we use enough
# (b) Prime cutoff X — yes truncation in primes! Let's test increasing X.
# Let me compute with X=10^4 vs 5e4 to see scaling

t1 = time.time()
Q5, comps5 = compute_Q_modular(zeros_float, T0, sigma, J, weight=12, conductor=1,
 primes_cutoff=50000, n_nodes_quad=100,
 include_negative_zeros=True)
print(f"X=5e4: {time.time()-t1:.1f}s")
trZ5, trA5 = np.trace(comps5['M_zeros']), np.trace(comps5['M_arith'])
print(f" tr(M_zeros)={trZ5:.6e} tr(M_arith)={trA5:.6e} diff={trZ5-trA5:.4e} rel={(trZ5-trA5)/trZ5:.4%}")
print(f" tr(M_primes)={np.trace(comps5['M_primes']):.6e}")


TimeoutError: Code execution timed out after 157.0 seconds